### RAG pipeline data ingestion to vector DB pipeline

#read document and data in document here first step of data ingestion

In [ ]:
import os
from pathlib import Path

from langchain_core.documents import Document
from pypdf import PdfReader


In [ ]:
def resolve_data_directory():
    """Find the project data directory regardless of the notebook working directory."""
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, current_dir.parent]:
        data_dir = candidate / "data"
        if data_dir.exists():
            return data_dir
    return current_dir / "data"


def process_documents(data_directory):
    """Process PDF and TXT files in a directory."""
    all_documents = []
    data_dir = Path(data_directory)
    if not data_dir.exists():
        print(f"Data directory not found: {data_dir}")
        return all_documents

    supported_files = sorted(list(data_dir.rglob("*.pdf")) + list(data_dir.rglob("*.txt")))
    print(f"Found {len(supported_files)} files to process")

    for file_path in supported_files:
        print(f"\nProcessing: {file_path.name}")
        try:
            if file_path.suffix.lower() == ".pdf":
                reader = PdfReader(str(file_path))
                page_texts = [page.extract_text() or "" for page in reader.pages]
                documents = [
                    Document(
                        page_content=text,
                        metadata={
                            "source_file": file_path.name,
                            "file_type": "pdf",
                            "source_path": str(file_path),
                            "page": index + 1,
                        },
                    )
                    for index, text in enumerate(page_texts)
                    if text.strip()
                ]
            else:
                text = file_path.read_text(encoding="utf-8", errors="ignore")
                documents = [
                    Document(
                        page_content=text,
                        metadata={
                            "source_file": file_path.name,
                            "file_type": "txt",
                            "source_path": str(file_path),
                        },
                    )
                ]

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} document segment(s)")
        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


data_dir = resolve_data_directory()
all_pdf_documents = process_documents(data_dir)


In [ ]:
all_pdf_documents

In [ ]:
### Text splitting get into chunks


def recursive_chunk_text(text, chunk_size=1000, chunk_overlap=200, separators=None):
    """Recursively split text into chunks using progressively smaller separators."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    text = text.strip()
    if len(text) <= chunk_size:
        return [text]

    separator = separators[0]
    if separator and separator in text:
        parts = [part.strip() for part in text.split(separator) if part.strip()]
        chunks = []
        for part in parts:
            if len(part) <= chunk_size:
                chunks.append(part)
            else:
                chunks.extend(recursive_chunk_text(part, chunk_size, chunk_overlap, separators[1:]))

        merged_chunks = []
        for chunk in chunks:
            if not merged_chunks:
                merged_chunks.append(chunk)
            elif len(merged_chunks[-1]) + len(separator) + len(chunk) <= chunk_size:
                merged_chunks[-1] = merged_chunks[-1] + separator + chunk
            else:
                merged_chunks.append(chunk)

        final_chunks = []
        for chunk in merged_chunks:
            if len(chunk) > chunk_size:
                for i in range(0, len(chunk), chunk_size - chunk_overlap):
                    final_chunks.append(chunk[i : i + chunk_size])
            else:
                final_chunks.append(chunk)
        return final_chunks

    if len(separators) > 1:
        return recursive_chunk_text(text, chunk_size, chunk_overlap, separators[1:])

    chunks = []
    for start in range(0, len(text), chunk_size - chunk_overlap):
        chunks.append(text[start : start + chunk_size])
    return chunks


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller, recursive chunks for better retrieval."""
    split_docs = []
    seen_texts = set()
    for doc in documents:
        text = doc.page_content or ""
        if not text.strip():
            continue

        chunks = recursive_chunk_text(text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        for chunk in chunks:
            normalized_chunk = chunk.strip()
            if not normalized_chunk or normalized_chunk in seen_texts:
                continue
            seen_texts.add(normalized_chunk)
            chunk_doc = Document(page_content=normalized_chunk, metadata=dict(doc.metadata))
            split_docs.append(chunk_doc)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [ ]:

chunks=split_documents(all_pdf_documents)
chunks

###embedding and vector store db

In [ ]:
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [ ]:
pip show sentence-transformers

In [ ]:
import re


class SimpleFallbackEmbeddingModel:
    """Simple deterministic embedding fallback used when sentence-transformers is unavailable."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name

    def get_sentence_embedding_dimension(self) -> int:
        return 32

    def encode(self, texts: List[str], show_progress_bar: bool = False) -> np.ndarray:
        if isinstance(texts, str):
            texts = [texts]

        embeddings = []
        for text in texts:
            tokens = re.findall(r"\w+", text.lower())
            vector = np.zeros(self.get_sentence_embedding_dimension(), dtype=np.float32)
            for token in tokens:
                index = sum(ord(ch) for ch in token) % self.get_sentence_embedding_dimension()
                vector[index] += 1.0

            norm = np.linalg.norm(vector)
            if norm > 0:
                vector = vector / norm
            embeddings.append(vector)

        return np.vstack(embeddings)


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer or a fallback."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model, falling back when needed."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            if SentenceTransformer is None:
                print("sentence-transformers is unavailable; using a lightweight fallback embedding model.")
                self.model = SimpleFallbackEmbeddingModel(self.model_name)
            else:
                self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            self.model = SimpleFallbackEmbeddingModel(self.model_name)
            print("Falling back to deterministic embeddings.")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

###vector store

In [ ]:
import os
import shutil
import uuid
from typing import List, Any

import chromadb
import numpy as np


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str | None = None):
        """
        Initialize the vector store. If `persist_directory` is None, create a unique
        per-run directory under `../data/vector_store_runs/` to avoid file-locking issues
        when removing previous stores.
        """
        if persist_directory is None:
            base = os.path.join("..", "data", "vector_store_runs")
            os.makedirs(base, exist_ok=True)
            persist_directory = os.path.join(base, uuid.uuid4().hex[:8])

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Persist directory: {self.persist_directory}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def reset_collection(self):
        """Remove all documents by removing the persistent directory and rebuilding a clean store."""
        try:
            if self.client is not None:
                try:
                    self.client.close()
                except Exception:
                    pass

            if os.path.exists(self.persist_directory):
                print(f"Resetting persistent vector store directory: {self.persist_directory}")
                shutil.rmtree(self.persist_directory)

            os.makedirs(self.persist_directory, exist_ok=True)
            self._initialize_store()
            print("Collection reset complete.")
        except Exception as e:
            print(f"Error resetting collection: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Create a fresh, unique vector store for this run
vectorstore = VectorStore()
# Do not auto-delete the store now (avoids permission issues); call reset_collection() manually if needed
vectorstore


In [ ]:
chunks

In [ ]:
###convert text to embedding

#extract text from chunk and create embedding

#generate embeddings

### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        query = (query or "").strip()
        if not query:
            print("Empty query provided to retriever")
            return []

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                retrieved_docs.sort(key=lambda item: item['similarity_score'], reverse=True)
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What is infobeans")

In [ ]:
rag_retriever.retrieve("founder of company infobeans")

### integration vectordb context pipeline with llm output

llm generate summarized output

In [ ]:
#simple rag pipeline with groq llm

import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

print(os.getenv("GROQ_API_KEY"))

In [ ]:
try:
    from langchain_groq import ChatGroq
except ImportError:
    ChatGroq = None

try:
    from langchain_core.prompts import PromptTemplate
except ImportError:
    from langchain.prompts import PromptTemplate

try:
    from langchain_core.messages import HumanMessage, SystemMessage
except ImportError:
    from langchain.schema import HumanMessage, SystemMessage

print("LangChain imports ready")


In [ ]:
from types import SimpleNamespace


class FallbackLLM:
    """Simple local fallback used when Groq is unavailable."""

    def __init__(self, model_name: str = "fallback"):
        self.model_name = model_name

    def invoke(self, payload):
        if isinstance(payload, list) and payload:
            content = payload[0].content if hasattr(payload[0], "content") else str(payload[0])
        else:
            content = str(payload)

        if "context" in content.lower() and "question" in content.lower():
            return SimpleNamespace(content="I couldn't access the Groq API, so I am using a fallback response. The retrieved context appears to be available locally.")
        return SimpleNamespace(content="I couldn't access the Groq API. Please provide a GROQ_API_KEY to enable live responses.")


class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str = None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if self.api_key and ChatGroq is not None:
            self.llm = ChatGroq(
                groq_api_key=self.api_key,
                model_name=self.model_name,
                temperature=0.1,
                max_tokens=1024,
            )
            self._use_fallback = False
            print(f"Initialized Groq LLM with model: {self.model_name}")
        else:
            self.llm = FallbackLLM(model_name=model_name)
            self._use_fallback = True
            print("Groq API key not found or ChatGroq is unavailable; using a local fallback response generator.")

    def invoke(self, messages):
        try:
            return self.llm.invoke(messages)
        except Exception as e:
            return SimpleNamespace(content=f"Error generating response: {e}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so.""",
        )

        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"


In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [ ]:
import os
print(os.getcwd())

In [ ]:
print(os.getenv("GROQ_API_KEY"))

In [ ]:


### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")


###Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
import os
from dotenv import load_dotenv
from pathlib import Path

root_dir = Path.cwd() if (Path.cwd() / ".env").exists() else Path.cwd().parent
load_dotenv(root_dir / ".env")

### Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = GroqLLM(api_key=groq_api_key)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return "No relevant context found to answer the question."

    context = "\n\n".join([f"[{doc['rank']}] {doc['content']}" for doc in results])
    return llm.generate_response(query, context)


In [ ]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

In [ ]:

from dotenv import load_dotenv
import os

load_dotenv("../.env")

key = os.getenv("GROQ_API_KEY")

print(key)
print(len(key) if key else "Key not found")

In [ ]:
print(repr(key))

In [ ]:
answer=rag_simple("What is toastmasters?",rag_retriever,llm)
print(answer)

###
Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "\n\n".join([f"[{doc['rank']}] {doc['content']}" for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("hard negative mining techniques", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])